# MOSAIC, Taken Apart: a Hands-On Primer
**Series B: Deep Learning for Molecules, Post 6c** &nbsp;|&nbsp; companion *primer* notebook

MOSAIC (Li, Sarkar, Lu, ..., Newhouse and Batista, *Nature* **651**, 107, 2026; code at github.com/batistagroup/MOSAIC) turns the open Llama-3.1-8B-Instruct language model into **2,498 specialists**, one for each Voronoi cell of a learned map of reaction space, and answers a query by consulting the nearest few. It generates full experimental procedures for reactions of new compounds, with a confidence estimate: the distance from the query to the specialist's cell centre.

> **Two parts.** **Part I** (everything up to the self-check) runs anywhere in about a minute. **Part II** (at the end) runs the authors' code and needs the authors' artifacts, a large GPU and the Llama weights; it has three levels, the first of which always runs.
>
> **What Part I is, and is not.** The real system needs the commercial Pistachio reaction database, GPUs and an 8-billion-parameter model, none of which are available here. Part I rebuilds the **mechanisms** in NumPy: on a *synthetic reaction universe*, a learned reaction metric, Voronoi cells with IVF-style search, specialists per cell, set-based matching, distance as confidence, the experiment-ranking rule; and *at the real size*, the kernel metric network (checked against its documented parameter count), a small Llama-style decoder with LoRA experts (checked against Llama-3.1-8B's parameter count and tested for the properties the system relies on), the training arithmetic, the prompt template, and diverse beam search. It does **not** run a language model and **does not reproduce MOSAIC's accuracy**; all toy numbers are for the toy. The paper's numbers (from the ChemRxiv preprint, not peer reviewed) are typed in where we analyse them. Information about the authors' code comes from their README and a machine-generated wiki of the repository (second-hand; we mark it).

| Section | What you do |
|---|---|
| 1 | build a synthetic universe of reactions with difference fingerprints |
| 2 | train a kernel metric network; compare the learned metric with raw fingerprints |
| 2a | the real kernel metric network at its real size: layers, input assembly, 1,901,549 parameters |
| 3 | partition the space into Voronoi cells (k-means) and search it IVF-style |
| 4 | build specialists, route queries, and score them with the paper's set metric |
| 5 | **a critical look at the metric:** more experts means bigger predicted sets |
| 6 | distance to the cell centre as a confidence |
| 7 | analyse the paper's Table 1 |
| 8 | the experiment-ranking rule |
| 9 | the paper's yield-prediction fit lines |
| 10 | scale arithmetic: Llama-3.1-8B, LoRA, and search cost |
| 10a-d | a complete Llama-style decoder with LoRA experts (tested); training arithmetic; the prompt template; diverse beam search |
| **Part II** | **run the authors' code and visualize a prediction (3 levels; needs their artifacts, a big GPU and the Llama weights for the upper levels)** |

## 0. Setup

In [ ]:
import itertools, math
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. A synthetic universe of reactions
MOSAIC encodes a reaction as concatenated **RDKit and Morgan fingerprints** of the reactants and of the product, plus a **difference fingerprint** (product minus reactants) that records *what changed*. We cannot run RDKit here, so we simulate the structure of this input. A reaction has a hidden **class** (40 classes in 8 families; classes of a family share part of their change pattern, like related coupling reactions). The reactant fingerprint is random substrate noise; the product is the reactant with the class's pattern of bits switched on and off, each bit applied with probability 0.65. The feature vector is $[r,\,p,\,p-r]$ (768 numbers).

In [ ]:
D, C, FAM = 256, 40, 8; PER = C // FAM
rng_u = np.random.default_rng(0)
fam_on  = [rng_u.choice(D, 8, replace=False) for _ in range(FAM)]
cls_on  = [np.r_[fam_on[c // PER], rng_u.choice(D, 4, replace=False)] for c in range(C)]       # bits a reaction of class c switches on
cls_off = [rng_u.choice(D, 4, replace=False) for c in range(C)]                                  # bits it switches off

def make_reaction(c, rng, keep=0.65, density=0.12):
    r = (rng.random(D) < density).astype(np.float32)                                              # reactant fingerprint: substrate noise
    r[cls_off[c]] = np.maximum(r[cls_off[c]], (rng.random(4) < 0.7))                              # the reactant carries the bits that will be removed
    p = r.copy(); p[cls_on[c][rng.random(len(cls_on[c])) < keep]] = 1; p[cls_off[c][rng.random(4) < keep]] = 0
    return np.r_[r, p, p - r]                                                                    # [reactant, product, difference]

def dataset(n, rng, prior, keep=0.65):
    y = rng.choice(C, n, p=prior); return np.stack([make_reaction(c, rng, keep) for c in y]), y

prior = rng_u.dirichlet(np.ones(C) * 1.5)                                                         # classes of very different sizes
Xtr, ytr = dataset(8000, rng_u, prior); Xte, yte = dataset(2000, rng_u, prior)
print(f"{len(Xtr)} training and {len(Xte)} test reactions, {C} classes in {FAM} families, {Xtr.shape[1]} features")
print(f"class sizes in the training set: min {np.bincount(ytr, minlength=C).min()}, max {np.bincount(ytr, minlength=C).max()}")
print(f"a typical difference fingerprint has {int((Xtr[0, 2 * D:] != 0).sum())} nonzero entries out of {D}")

## 2. The kernel metric network (KMN)
The paper trains a network to classify reactions into **2,285 classes** and takes the **128-dimensional layer before the output** as the *Reaction-Specific Fingerprint* (RSFP). The Euclidean distance between RSFPs then plays the role of a similarity kernel (Eq. 1 of the paper). Here: a small network $768\to256\to128\to40$ trained with softmax cross-entropy.
**Stop and predict:** how much better than the raw fingerprints will a nearest-neighbour search be in RSFP space?

In [ ]:
def init_kmn(rng):
    s = lambda a, b: (rng.normal(size=(a, b)) * np.sqrt(2 / a)).astype(np.float32)
    return dict(W1=s(3 * D, 256), b1=np.zeros(256, np.float32), W2=s(256, 128), b2=np.zeros(128, np.float32), W3=s(128, C) * 0.5, b3=np.zeros(C, np.float32))

def forward(P, X):
    h = np.maximum(X @ P["W1"] + P["b1"], 0); z = np.tanh(h @ P["W2"] + P["b2"])                  # z is the 128-dim RSFP
    return h, z, z @ P["W3"] + P["b3"]

def train_kmn(P, X, y, epochs=12, bs=128, lr=2e-3, seed=1):
    r = np.random.default_rng(seed); m = {k: np.zeros_like(v) for k, v in P.items()}; v = {k: np.zeros_like(x) for k, x in P.items()}; t = 0
    for ep in range(epochs):
        idx = r.permutation(len(X))
        for i in range(0, len(X), bs):
            b = idx[i:i + bs]; xb, yb = X[b], y[b]; h, z, lg = forward(P, xb)
            p = np.exp(lg - lg.max(1, keepdims=True)); p /= p.sum(1, keepdims=True); g = p; g[np.arange(len(b)), yb] -= 1; g /= len(b)
            gpre2 = (g @ P["W3"].T) * (1 - z ** 2); gh = gpre2 @ P["W2"].T; gpre1 = gh * (h > 0)
            grads = dict(W1=xb.T @ gpre1, b1=gpre1.sum(0), W2=h.T @ gpre2, b2=gpre2.sum(0), W3=z.T @ g, b3=g.sum(0)); t += 1
            for k in P:
                m[k] = 0.9 * m[k] + 0.1 * grads[k]; v[k] = 0.999 * v[k] + 0.001 * grads[k] ** 2
                P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return P

kmn = train_kmn(init_kmn(np.random.default_rng(2)), Xtr, ytr)
_, ztr, _ = forward(kmn, Xtr); _, zte, lgte = forward(kmn, Xte)
print(f"classification accuracy on test reactions: {(lgte.argmax(1) == yte).mean():.3f}")

def nn_accuracy(A, ya, B, yb):                                                  # 1-nearest-neighbour class accuracy
    d = (B ** 2).sum(1)[:, None] + (A ** 2).sum(1)[None, :] - 2 * B @ A.T; return (ya[d.argmin(1)] == yb).mean()

print("\n1-nearest-neighbour class accuracy (Euclidean distance):")
print(f"   raw [reactant, product, difference] : {nn_accuracy(Xtr, ytr, Xte, yte):.3f}")
print(f"   difference fingerprint only          : {nn_accuracy(Xtr[:, 2 * D:], ytr, Xte[:, 2 * D:], yte):.3f}")
print(f"   learned RSFP (128 numbers)           : {nn_accuracy(ztr, ytr, zte, yte):.3f}")

In [ ]:
# a 2-D view of the learned space (first two principal components), coloured by reaction family
Zc = ztr - ztr.mean(0); U, S, Vt = np.linalg.svd(Zc[:3000], full_matrices=False); pc = Zc @ Vt[:2].T
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
cm = plt.cm.tab10
for f in range(FAM):
    sel = (ytr // PER) == f; ax[0].scatter(pc[sel, 0], pc[sel, 1], s=3, color=cm(f), label=f"family {f}")
ax[0].set(xlabel="PC 1", ylabel="PC 2", title="(a) RSFP space: related reactions cluster"); ax[0].legend(fontsize=6, ncol=2, markerscale=3)
Xc = Xtr[:, 2 * D:] - Xtr[:, 2 * D:].mean(0); U2, S2, Vt2 = np.linalg.svd(Xc[:3000], full_matrices=False); pr = Xc @ Vt2[:2].T
for f in range(FAM):
    sel = (ytr // PER) == f; ax[1].scatter(pr[sel, 0], pr[sel, 1], s=3, color=cm(f))
ax[1].set(xlabel="PC 1", ylabel="PC 2", title="(b) The same view of the raw difference fingerprints")
plt.tight_layout(); plt.show()

## 2a. The real KMN, at its real size
The toy network above is a stand-in. The authors' KMN is documented (by the repository's machine-generated wiki, which we mark as second-hand) as a three-layer perceptron: **$6144\\to256\\to128\\to2285$**, with ReLU, batch normalization and 30 % dropout after each hidden layer. The input is a *mixed* fingerprint of the reaction: for each of **reactant, difference and product**, a 1024-bit RDKit fingerprint and a 1024-bit Morgan fingerprint (2048 numbers), three times: $3\\times2048=6144$. (The wiki's text says "2048 bits $\\times3$" for each kind, which would give 12,288; the total of 6144 and the `fp_size=1024` it quotes for the code fit our reading.) The 2285 output classes are the distinct **reaction names** of the training data; the **RSFP is the 128-number output of the second hidden layer**, taken in evaluation mode (no dropout, batch-norm running statistics). The wiki states a total of **1,901,549 parameters**. We build the network with random weights and check that count.
**Stop and predict:** which layer holds most of the parameters?

In [ ]:
D_IN, H1, H2, N_CLASSES = 6144, 256, 128, 2285

class RealSizeKMN:
    """Linear(6144,256)-ReLU-BatchNorm-Dropout(0.3)-Linear(256,128)-ReLU-BatchNorm-Dropout(0.3)-Linear(128,2285); random weights."""
    def __init__(self, seed=0):
        r = np.random.default_rng(seed); w = lambda a, b: (r.normal(size=(a, b)) * np.sqrt(2 / a)).astype(np.float32); z = lambda n: np.zeros(n, np.float32); o = lambda n: np.ones(n, np.float32)
        self.p = dict(W1=w(D_IN, H1), b1=z(H1), g1=o(H1), be1=z(H1), W2=w(H1, H2), b2=z(H2), g2=o(H2), be2=z(H2), W3=w(H2, N_CLASSES), b3=z(N_CLASSES))
        self.buf = dict(m1=z(H1), v1=o(H1), m2=z(H2), v2=o(H2))                       # batch-norm running statistics: buffers, not parameters
    def n_params(self): return sum(v.size for v in self.p.values())
    @staticmethod
    def _bn(x, g, b, m, v, eps=1e-5): return (x - m) / np.sqrt(v + eps) * g + b
    def get_embeddings(self, x):                                                        # evaluation mode: no dropout, running statistics -> the 128-number RSFP
        P, B = self.p, self.buf
        h = self._bn(np.maximum(x @ P["W1"] + P["b1"], 0), P["g1"], P["be1"], B["m1"], B["v1"])
        return self._bn(np.maximum(h @ P["W2"] + P["b2"], 0), P["g2"], P["be2"], B["m2"], B["v2"])
    def logits(self, x): return self.get_embeddings(x) @ self.p["W3"] + self.p["b3"]
    def forward_train(self, x, rng, drop=0.3, momentum=0.1):                          # training mode: batch statistics, random dropout, running statistics updated
        P, B = self.p, self.buf; h = np.maximum(x @ P["W1"] + P["b1"], 0); m, v = h.mean(0), h.var(0)
        B["m1"] = (1 - momentum) * B["m1"] + momentum * m; B["v1"] = (1 - momentum) * B["v1"] + momentum * v
        h = (h - m) / np.sqrt(v + 1e-5) * P["g1"] + P["be1"]; h = h * (rng.random(h.shape) > drop) / (1 - drop)
        h = np.maximum(h @ P["W2"] + P["b2"], 0); m, v = h.mean(0), h.var(0)
        h = (h - m) / np.sqrt(v + 1e-5) * P["g2"] + P["be2"]; h = h * (rng.random(h.shape) > drop) / (1 - drop)
        return h @ P["W3"] + P["b3"]

kmn_real = RealSizeKMN()
rows = [("fc1  6144 -> 256", kmn_real.p["W1"].size + kmn_real.p["b1"].size), ("fc2   256 -> 128", kmn_real.p["W2"].size + kmn_real.p["b2"].size),
        ("fc3   128 -> 2285", kmn_real.p["W3"].size + kmn_real.p["b3"].size), ("batch norm 1 (scale, shift)", 2 * H1), ("batch norm 2 (scale, shift)", 2 * H2)]
tot = kmn_real.n_params()
print(pd.DataFrame([(n, f"{c:,}", f"{100 * c / tot:.1f} %") for n, c in rows], columns=["layer", "parameters", "share"]).to_string(index=False))
print(f"\ntotal {tot:,} parameters (the wiki states 1,901,549; its table row for fc3 says 294,485, an arithmetic slip: 128 x 2285 + 2285 = {rows[2][1]:,}); a float32 file of {4 * tot / 1e6:.1f} MB")
assert tot == 1_901_549
print(f"the input layer holds {100 * rows[0][1] / tot:.0f} % of the parameters (it is as wide as the fingerprint); the whole KMN is {8_030_261_248 / tot:,.0f} times smaller than Llama-3.1-8B")

def assemble_input(rdkit_r, morgan_r, rdkit_p, morgan_p):
    """The documented input: [reactant | difference | product], each = 1024 RDKit bits + 1024 Morgan bits."""
    r = np.r_[rdkit_r, morgan_r].astype(np.int8); p = np.r_[rdkit_p, morgan_p].astype(np.int8)
    return np.r_[r, p - r, p]

rr = np.random.default_rng(3); bits = lambda: (rr.random(1024) < 0.05).astype(np.int8)
x_one = assemble_input(bits(), bits(), bits(), bits()); print("assembled input:", x_one.shape, "values in", sorted(set(x_one.tolist())), "(the difference block can be -1, 0 or +1)")
X_batch = np.stack([assemble_input(bits(), bits(), bits(), bits()) for _ in range(64)]).astype(np.float32)
e1, e2 = kmn_real.get_embeddings(X_batch), kmn_real.get_embeddings(X_batch)
print("RSFP of a batch:", e1.shape, "| evaluation mode is deterministic:", np.array_equal(e1, e2))
t1, t2 = kmn_real.forward_train(X_batch, np.random.default_rng(0)), kmn_real.forward_train(X_batch, np.random.default_rng(1))
print(f"training mode (dropout) gives different logits on the same input: max difference {np.abs(t1 - t2).max():.2f}; running statistics moved: {np.abs(kmn_real.buf['m1']).max() > 0}")
assert x_one.shape == (6144,) and e1.shape == (64, 128) and np.array_equal(e1, e2) and np.abs(t1 - t2).max() > 0.1

## 3. Voronoi cells and IVF search
MOSAIC clusters all RSFPs with FAISS's inverted-file index (IVF), whose coarse quantizer is **k-means**: the 128-dimensional space is cut into **Voronoi cells** around cluster centres. Each cell becomes one specialist's domain (2,500 cells in the paper; 30 here). A query is encoded, compared with the centres, and only the closest cell(s) are searched in detail.

In [ ]:
def kmeans(Z, K, rng, iters=30):
    cen = Z[rng.choice(len(Z), K, replace=False)].copy()
    for _ in range(iters):
        d = (Z ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * Z @ cen.T; a = d.argmin(1)
        for k in range(K):
            if (a == k).any(): cen[k] = Z[a == k].mean(0)
    return cen, a

K = 30
cen, atr = kmeans(ztr, K, np.random.default_rng(3))
sizes = np.bincount(atr, minlength=K)
print(f"{K} Voronoi cells; reactions per cell: min {sizes.min()}, median {int(np.median(sizes))}, max {sizes.max()}; empty cells: {(sizes == 0).sum()}")
purity = np.array([np.bincount(ytr[atr == k], minlength=C).max() / sizes[k] for k in range(K)])
fam_pur = np.array([np.bincount(ytr[atr == k] // PER, minlength=FAM).max() / sizes[k] for k in range(K)])
print(f"share of the dominant CLASS in a cell: mean {purity.mean():.2f} (min {purity.min():.2f}); of the dominant FAMILY: mean {fam_pur.mean():.2f}")

k0 = int(np.argmin(purity)); comp = np.bincount(ytr[atr == k0], minlength=C) / sizes[k0]; top = np.argsort(comp)[::-1][:3]
print(f"\nthe most mixed cell ({k0}, {sizes[k0]} reactions): " + ", ".join(f"class {c} ({100 * comp[c]:.1f} %)" for c in top))
print("(the paper's Fig. 1e: expert 883 is 70.69 % chloro Buchwald-Hartwig, 17.22 % chloro SNAr, 0.26 % Goldberg-type: related reactions share a cell)")

In [ ]:
# IVF search: probe only the nprobe nearest cells instead of every reaction
def ivf_search(zq, nprobe):
    dc = (zq ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * zq @ cen.T; order = np.argsort(dc, 1)
    found, cost = [], []
    for i in range(len(zq)):
        cand = np.where(np.isin(atr, order[i, :nprobe]))[0]
        d = ((ztr[cand] - zq[i]) ** 2).sum(1); found.append(cand[d.argmin()]); cost.append(K + len(cand))
    return np.array(found), np.array(cost)

d_all = (zte ** 2).sum(1)[:, None] + (ztr ** 2).sum(1)[None, :] - 2 * zte @ ztr.T; exact_nn = d_all.argmin(1)
print(f"{'nprobe':>6} | {'recall of the true nearest neighbour':>37} | {'distance computations / brute force':>36}")
for npb in (1, 2, 3, 5):
    f, cost = ivf_search(zte, npb); print(f"{npb:>6} | {np.mean(f == exact_nn):>37.3f} | {cost.mean():>15.0f} / {len(ztr)}  ({100 * cost.mean() / len(ztr):.1f} %)")

## 4. Specialists, routing and the paper's set metric
We need something for each specialist to predict. We invent **conditions**: each class has three typical (reagent set, solvent) combinations with probabilities 0.55 / 0.30 / 0.15 (related classes share reagents); a reaction's true conditions are drawn from them, with 15 % chance of one extra reagent and 10 % chance of a random solvent. A toy "specialist" for a cell simply lists the condition sets most frequent among its training reactions (the analogue of the model's top beam).
The paper scores predicted against true sets of reagents or solvents with
$$D=1-\frac{|S_{\rm pred}\cap S_{\rm true}|}{|S_{\rm true}|}:$$
$D=0$ means the **true set is a subset of the predicted set** ("exact match"); $D<1$ means at least one true molecule was predicted ("partial match"). Predictions come from the closest expert (*one-shot*) or from up to three nearest experts (*multiple-shots*).

In [ ]:
NR, NS = 30, 10
rc = np.random.default_rng(10)
fam_reag = [rc.choice(NR, 3, replace=False) for _ in range(FAM)]
typical = [[(frozenset({int(rc.choice(fam_reag[c // PER])), int(rc.integers(NR))}), int(rc.integers(NS))) for _ in range(3)] for c in range(C)]

def true_conditions(c, rng):
    reag, sol = typical[c][rng.choice(3, p=[0.55, 0.30, 0.15])]; reag = set(reag)
    if rng.random() < 0.15: reag.add(int(rng.integers(NR)))                    # an extra additive
    if rng.random() < 0.10: sol = int(rng.integers(NS))                         # an unusual solvent
    return frozenset(reag), sol

rng_c = np.random.default_rng(11)
cond_tr = [true_conditions(c, rng_c) for c in ytr]
keep_te = np.where(np.random.default_rng(12).random(len(yte)) < 0.8, 0.65, 0.35)       # 20 % of test reactions have a weaker class signal (harder, less typical)
Xte2 = np.stack([make_reaction(c, np.random.default_rng(1000 + i), keep=k) for i, (c, k) in enumerate(zip(yte, keep_te))])
_, zte2, _ = forward(kmn, Xte2); cond_te = [true_conditions(c, rng_c) for c in yte]

experts = []                                                                    # per cell: condition sets ordered by frequency (the 'beam')
for k in range(K):
    cnt = Counter(cond_tr[i] for i in np.where(atr == k)[0]); experts.append([s for s, _ in cnt.most_common()])

dist = (zte2 ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * zte2 @ cen.T       # squared Euclidean distance to every centre, as FAISS returns
order = np.argsort(dist, 1)

def D_metric(pred, true): return 1 - len(pred & true) / len(true)

def predict(i, n_experts):                                                         # top-1 prediction of each of the n nearest experts, pooled
    reag, sol = set(), set()
    for k in order[i, :n_experts]:
        if experts[k]: reag |= set(experts[k][0][0]); sol.add(experts[k][0][1])
    return reag, sol

def evaluate(n_experts):
    out = {"reagent": [], "solvent": [], "both": []}; size = []
    for i in range(len(zte2)):
        pr, ps = predict(i, n_experts); tr, ts = cond_te[i]; size.append(len(pr) + len(ps))
        out["reagent"].append(D_metric(pr, set(tr))); out["solvent"].append(D_metric(ps, {ts}))
        out["both"].append(D_metric(pr | {("s", x) for x in ps}, set(tr) | {("s", ts)}))
    return {k: (100 * np.mean(np.array(v) == 0), 100 * np.mean(np.array(v) < 1)) for k, v in out.items()}, float(np.mean(size))

rows = []
for n, label in ((1, "one-shot (nearest expert)"), (3, "multiple-shots (3 nearest experts)")):
    res, _ = evaluate(n)
    rows.append((label, *[f"{res[k][0]:.1f} / {res[k][1]:.1f}" for k in ("reagent", "solvent", "both")]))
print("toy universe: exact / partial match rates (%)")
print(pd.DataFrame(rows, columns=["", "reagent", "solvent", "both"]).to_string(index=False))
print("\n(the paper's preprint reports, for real reactions: one-shot reagent 22.4 / 45.4, solvent 29.8 / 51.7, both 12.9 / 73.0; multiple-shots 43.0 / 76.0, 32.8 / 55.2, 28.9 / 94.8)")

## 5. A critical look at "exact match"
Because $D=0$ whenever the true set is *contained* in the prediction, a prediction that lists more molecules can only do better. Using up to three experts enlarges the predicted set, so part of the improvement from "one-shot" to "multiple-shots" is mechanical. The metric ignores **precision**. We measure both: how the exact-match rate, the size of the predicted set and the precision of the reagent prediction change as we pool more experts, and what a degenerate predictor that lists *every* reagent would score.
**Stop and predict:** what does the degenerate predictor score?

In [ ]:
prec_rows = []
for n in (1, 2, 3, 5, 8):
    ex, prec, size = [], [], []
    for i in range(len(zte2)):
        pr, _ = predict(i, n); tr = set(cond_te[i][0]); ex.append(D_metric(pr, tr) == 0); size.append(len(pr)); prec.append(len(pr & tr) / max(len(pr), 1))
    prec_rows.append((f"{n} expert(s)", 100 * np.mean(ex), np.mean(size), 100 * np.mean(prec)))
everything = set(range(NR)); ex_all = np.mean([D_metric(everything, set(c[0])) == 0 for c in cond_te])
prec_rows.append(("predict ALL 30 reagents", 100 * ex_all, float(NR), 100 * np.mean([len(set(c[0])) / NR for c in cond_te])))
print(pd.DataFrame(prec_rows, columns=["predictor", "exact-match rate (%)", "mean size of predicted reagent set", "precision (%)"]).round(1).to_string(index=False))
print("\nthe exact-match rate rises with every added expert, but so does the number of molecules the chemist has to choose among, and precision falls.")
print("The degenerate predictor is 'exact' 100 % of the time. A fair reading of Table 1 needs the size of the predicted sets, or precision, next to the match rates.")

## 6. Distance to the cell centre as confidence
MOSAIC's confidence is the **distance from the query's RSFP to the centroid** of the expert that answers: small means the query lies well inside the expert's domain. (FAISS returns *squared* Euclidean distances, which is presumably the scale of the paper's thresholds: below 50 high confidence, 100 to 200 moderate, above 200 only the broad reaction category.) Our toy distances have their own scale, so we look at the *shape* of the relationship.

In [ ]:
d1 = dist[np.arange(len(zte2)), order[:, 0]]
exact1 = np.array([D_metric(set(experts[order[i, 0]][0][0]) if experts[order[i, 0]] else set(), set(cond_te[i][0])) == 0 for i in range(len(zte2))], float)
edges = np.quantile(d1, np.linspace(0, 1, 6)); table = []
for a, b in zip(edges[:-1], edges[1:]):
    m = (d1 >= a) & (d1 <= b); table.append((f"{a:.1f} to {b:.1f}", int(m.sum()), 100 * exact1[m].mean()))
print(pd.DataFrame(table, columns=["squared distance to the nearest centre", "reactions", "one-shot exact reagent match (%)"]).round(1).to_string(index=False))
rho, _ = spearmanr(d1, exact1); print(f"\nSpearman correlation between distance and success: {rho:.2f}")
weak = keep_te < 0.5
print(f"mean squared distance: typical reactions {d1[~weak].mean():.1f}, reactions with a weak class signal {d1[weak].mean():.1f}")
print("Even the closest quintile succeeds only about half the time: the true conditions are genuinely multi-modal (0.55 / 0.30 / 0.15 here). A small distance means 'in the domain', not 'right'.")

plt.figure(figsize=(6.5, 3.8)); plt.bar(range(5), [t[2] for t in table], color=COLORS["blue"]); plt.xticks(range(5), ["closest", "2", "3", "4", "farthest"])
plt.xlabel("quintile of distance to the nearest centre"); plt.ylabel("one-shot exact reagent match (%)"); plt.tight_layout(); plt.show()

## 7. Analysing the paper's Table 1
Typed in from the ChemRxiv preprint (reagent and solvent prediction on held-out reactions; "both" treats reagents and solvents as one set). Exact means $D=0$, partial means $D<1$.

In [ ]:
t1 = pd.DataFrame({"reagent": [22.4, 45.4, 43.0, 76.0], "solvent": [29.8, 51.7, 32.8, 55.2], "both": [12.9, 73.0, 28.9, 94.8]},
                  index=pd.MultiIndex.from_product([["one-shot", "multiple-shots"], ["exact", "partial"]]))
print(t1.to_string())
g = (t1.loc["multiple-shots"] / t1.loc["one-shot"]).round(2); print("\nmultiple-shots / one-shot:\n", g.to_string())
fail1, failn = 100 - t1.loc[("one-shot", "partial"), "both"], 100 - t1.loc[("multiple-shots", "partial"), "both"]
print(f"\nreactions where NOT ONE correct reagent or solvent is predicted: {fail1:.1f} % (one-shot) -> {failn:.1f} % (multiple-shots), a factor {fail1 / failn:.1f} fewer")
print("consistency checks: partial >= exact everywhere:", bool((t1.xs('partial', level=1).values >= t1.xs('exact', level=1).values).all()),
      "| 'both' partial >= reagent partial and solvent partial:", bool((t1.xs('partial', level=1)['both'] >= t1.xs('partial', level=1)[['reagent', 'solvent']].max(axis=1)).all()))
print("solvents gain much less from extra experts than reagents (exact x1.10 vs x1.92): there are fewer plausible solvents, so one expert already covers most of them")

# a crude model: each of 3 experts covers the truth independently with the one-shot probability q.  Then 1 - (1 - q)^3 would be the multiple-shots rate.
# Solve 1 - (1 - q)^m = observed for m: the EFFECTIVE number of independent experts that the three nearest experts amount to.
eff = {}
for col in t1.columns:
    for kind in ("exact", "partial"):
        q, obs = t1.loc[("one-shot", kind), col] / 100, t1.loc[("multiple-shots", kind), col] / 100
        eff[(col, kind)] = (1 - (1 - q) ** 3, obs, math.log(1 - obs) / math.log(1 - q))
print(pd.DataFrame(eff, index=["independence prediction", "observed", "effective independent experts (of 3)"]).T.round(2).to_string())
print("reagents behave like about 2.2 independent experts, solvents like about 1.1: the three experts nearly always suggest the same solvent")

## 8. The experiment-ranking rule
With $N$ experts and expert $e$ offering $M_e$ predictions, the paper ranks the experiment (expert $e$, prediction $p$) by
$$R(e,p)=N\,(p-1)+e,$$
so that every expert's first prediction is tried before any expert's second.

In [ ]:
def ranking(M):                                                         # M[e-1] = number of predictions of expert e
    N = len(M); return sorted(((N * (p - 1) + e, e, p) for e in range(1, N + 1) for p in range(1, M[e - 1] + 1)))

M = [4, 2, 3]
rk = ranking(M)
print("N = 3 experts offering", M, "predictions:")
print(pd.DataFrame(rk, columns=["rank R", "expert e", "prediction p"]).to_string(index=False))
ranks = [r for r, _, _ in rk]
print("\nranks unique:", len(set(ranks)) == len(ranks), "| all first predictions before any second prediction:",
      max(r for r, e, p in rk if p == 1) < min(r for r, e, p in rk if p == 2), "| ranks are dense (no gaps):", ranks == list(range(1, len(ranks) + 1)))
print("expert 2 has run out after two predictions, so ranks such as 8 and 11 never occur: the ranks order the experiments but are not consecutive")

## 9. The yield-prediction fit lines
The model writes the yield as text. To evaluate it, predictions are binned into ten intervals of 10 percentage points, and the figure of the preprint plots the median predicted yield of each bin against the average true yield of its members, with fit lines labelled $y=0.61x+24.99$ and $y=0.61x+24.93$ (the two panels: classes capped at 20 examples, and unconstrained).
**Stop and predict:** at which predicted yield does the fit say prediction and truth agree?

In [ ]:
a, b = 0.61, 24.99
fixed_point = b / (1 - a)
print(f"y = {a} x + {b} crosses y = x at x = {fixed_point:.1f} %: below it the model is too pessimistic on average, above it too optimistic")
centres = np.arange(5, 100, 10)
print(pd.DataFrame({"predicted yield, bin centre (%)": centres, "average true yield from the fit (%)": a * centres + b}).round(1).to_string(index=False))
print(f"\nthe slope {a} < 1 means the predictions are informative but over-dispersed: a predicted spread of 90 points corresponds to about {a * 90:.0f} true points.")
print(f"a linear recalibration x -> (y - {b}) / {a} would remove this compression (it cannot remove the scatter within each bin, which the preprint shows only in a figure)")

## 10. Scale arithmetic: Llama-3.1-8B, LoRA and search cost
Each expert is the Llama-3.1-8B-Instruct model plus a **LoRA adapter**: for a weight matrix $W\in\mathbb R^{d_{\rm out}\times d_{\rm in}}$, instead of retraining $W$ one learns $W'=W+\tfrac\alpha r BA$ with $B\in\mathbb R^{d_{\rm out}\times r}$, $A\in\mathbb R^{r\times d_{\rm in}}$ and rank $r\ll d$, i.e. $r\,(d_{\rm in}+d_{\rm out})$ parameters. **The preprint does not state the rank or the target layers.** The repository's machine-generated wiki (second-hand) documents rank 16, $\\alpha=64$ and adapters on all attention and feed-forward projections; the table below includes that configuration, and Section 10a looks at it more closely.

In [ ]:
# Llama-3.1-8B architecture (public model card values)
h, layers, inter, heads, kv_heads, vocab = 4096, 32, 14336, 32, 8, 128_256
head_dim = h // heads; kv = kv_heads * head_dim
attn = h * h + h * kv + h * kv + h * h                              # q, k, v, o projections
mlp = 3 * h * inter                                                 # gate, up, down projections
base_params = vocab * h * 2 + layers * (attn + mlp + 2 * h) + h     # embedding, output layer, blocks, norms
print(f"Llama-3.1-8B parameters from the architecture: {base_params / 1e9:.2f} billion ({2 * base_params / 1e9:.1f} GB in 16-bit)")

shapes = {"q": (h, h), "k": (h, kv), "v": (h, kv), "o": (h, h), "gate": (h, inter), "up": (h, inter), "down": (inter, h)}
def lora_params(r, targets):
    return layers * sum(r * (shapes[t][0] + shapes[t][1]) for t in targets)

N_EXPERTS = 2498
rows = []
for r, targets, label in ((8, "qkvo", "r = 8, attention only"), (16, "qkvo", "r = 16, attention only"), (16, ("q", "k", "v", "o", "gate", "up", "down"), "r = 16, all linear layers")):
    p = lora_params(r, targets); gb = 2 * p / 1e6
    rows.append((label, f"{p / 1e6:.1f} M", f"{100 * p / base_params:.2f} %", f"{gb:.0f} MB", f"{N_EXPERTS * gb / 1e3:.0f} GB"))
print(pd.DataFrame(rows, columns=["illustrative adapter", "parameters", "share of the base model", "size (16-bit)", f"{N_EXPERTS} adapters"]).to_string(index=False))
print(f"\nversus {N_EXPERTS} full copies of the model: {N_EXPERTS * 2 * base_params / 1e12:.0f} TB. Experts differ only by their adapter, so serving one means loading ~{2 * lora_params(16, ('q','k','v','o','gate','up','down')) / 1e6:.0f} MB, not 16 GB.")

In [ ]:
# training data per expert and the cost of searching the map: a rough, DERIVED estimate
# the preprint says its Reaxys/SciFinder-scale target (>100 million records) is 'nearly 100 times' the current training data
N_train = 100e6 / 100; cells = 2500
print(f"training reactions ~ {N_train / 1e6:.0f} million -> about {N_train / cells:.0f} reactions per Voronoi cell on average (cells are very unequal in size)")
for nprobe in (1, 3):
    ivf = cells + nprobe * N_train / cells
    print(f"IVF search with nprobe = {nprobe}: ~{ivf:,.0f} distance computations vs {N_train:,.0f} for brute force ({N_train / ivf:,.0f}x fewer)")
print("(the toy of Section 3 showed the same trade-off: recall of the exact nearest neighbour versus the fraction of the data scanned)")

## 10a. A complete Llama-style decoder with LoRA experts
Each MOSAIC expert is **Llama-3.1-8B-Instruct plus a LoRA adapter**. To describe the model completely we build a *small* decoder with Llama's structure and run it: **RMSNorm** instead of layer norm, **rotary position embeddings (RoPE)** applied to queries and keys, **grouped-query attention (GQA)** where several query heads share one key-value head, a **SwiGLU** feed-forward network $W_{\\rm down}\\,(\\mathrm{silu}(W_{\\rm gate}x)\\odot W_{\\rm up}x)$, and untied input and output embeddings. At full size, these formulas must give Llama-3.1-8B's published parameter count (8,030,261,248), which we check. Then we attach LoRA adapters to all seven linear layers of every block and test the properties the system relies on: a fresh adapter changes nothing; an adapter can be merged into the weights without changing the output; swapping adapters swaps the expert and leaves the base model untouched; and cached decoding gives the same logits as the full pass.
**Stop and predict:** how many parameters does a rank-16 adapter on all seven linear layers add to Llama-3.1-8B?

In [ ]:
def llama_param_count(h, layers, inter, heads, kv_heads, vocab, tie=False):
    kv = kv_heads * (h // heads); attn = 2 * h * h + 2 * h * kv; mlp = 3 * h * inter
    return vocab * h * (1 if tie else 2) + layers * (attn + mlp + 2 * h) + h

LLAMA31_8B = dict(h=4096, layers=32, inter=14336, heads=32, kv_heads=8, vocab=128_256)
print(f"Llama-3.1-8B from the architecture: {llama_param_count(**LLAMA31_8B):,} parameters")
assert llama_param_count(**LLAMA31_8B) == 8_030_261_248

def lora_count(cfg, r, targets=("q", "k", "v", "o", "gate", "up", "down")):
    h, kv = cfg["h"], cfg["kv_heads"] * (cfg["h"] // cfg["heads"]); sh = dict(q=(h, h), k=(h, kv), v=(h, kv), o=(h, h), gate=(h, cfg["inter"]), up=(h, cfg["inter"]), down=(cfg["inter"], h))
    return cfg["layers"] * sum(r * (sh[t][0] + sh[t][1]) for t in targets)
print(f"rank-16 LoRA on all seven linear layers: {lora_count(LLAMA31_8B, 16):,} parameters ({100 * lora_count(LLAMA31_8B, 16) / llama_param_count(**LLAMA31_8B):.2f} % of the base model)")

TINY = dict(h=64, layers=2, inter=176, heads=8, kv_heads=2, vocab=60)

def rms_norm(x, w, eps=1e-5): return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + eps) * w
silu = lambda x: x / (1 + np.exp(-x))

def rope(x, pos, base=500_000.0):                       # x: (L, heads, head_dim); rotate pairs (i, i + hd/2) by pos * theta_i (Llama-3 uses base 500000)
    hd = x.shape[-1]; half = hd // 2; theta = base ** (-np.arange(half) / half); ang = pos[:, None] * theta[None, :]
    c, s = np.cos(ang)[:, None, :], np.sin(ang)[:, None, :]; x1, x2 = x[..., :half], x[..., half:]
    return np.concatenate([x1 * c - x2 * s, x2 * c + x1 * s], -1)

class TinyLlama:
    def __init__(self, cfg=TINY, seed=0):
        r = np.random.default_rng(seed); s = lambda *sh: r.normal(size=sh) * 0.05; self.cfg = cfg; h = cfg["h"]; hd = h // cfg["heads"]; kv = cfg["kv_heads"] * hd
        self.emb = s(cfg["vocab"], h); self.head = s(h, cfg["vocab"]); self.norm = np.ones(h)
        self.blocks = [dict(q=s(h, h), k=s(h, kv), v=s(h, kv), o=s(h, h), gate=s(h, cfg["inter"]), up=s(h, cfg["inter"]), down=s(cfg["inter"], h), n1=np.ones(h), n2=np.ones(h)) for _ in range(cfg["layers"])]
    def n_params(self):
        return self.emb.size + self.head.size + self.norm.size + sum(v.size for b in self.blocks for v in b.values())
    def new_adapter(self, r=4, alpha=16, seed=1, zero_B=True, targets=("q", "k", "v", "o", "gate", "up", "down")):
        g = np.random.default_rng(seed); ad = dict(r=r, alpha=alpha, layers=[])
        for b in self.blocks:
            ad["layers"].append({t: (g.normal(size=(b[t].shape[0], r)) * 0.1, np.zeros((r, b[t].shape[1])) if zero_B else g.normal(size=(r, b[t].shape[1])) * 0.1) for t in targets})
        return ad
    @staticmethod
    def lin(x, W, ad, name):                                                          # y = x W  +  (alpha / r) (x A) B   (LoRA)
        y = x @ W
        if ad is not None and name in ad: A, B = ad[name]; y = y + (ad["__scale__"]) * ((x @ A) @ B)
        return y
    def forward(self, ids, adapter=None, k_cache=None, v_cache=None, start=0):
        cfg = self.cfg; h = cfg["h"]; nh, nkv = cfg["heads"], cfg["kv_heads"]; hd = h // nh; grp = nh // nkv; L = len(ids); pos = np.arange(start, start + L).astype(float)
        x = self.emb[ids]
        for li, b in enumerate(self.blocks):
            ad = None
            if adapter is not None: ad = dict(adapter["layers"][li]); ad["__scale__"] = adapter["alpha"] / adapter["r"]
            y = rms_norm(x, b["n1"])
            q = rope(self.lin(y, b["q"], ad, "q").reshape(L, nh, hd), pos); k = rope(self.lin(y, b["k"], ad, "k").reshape(L, nkv, hd), pos); v = self.lin(y, b["v"], ad, "v").reshape(L, nkv, hd)
            if k_cache is not None:
                k_cache[li] = k if k_cache[li] is None else np.concatenate([k_cache[li], k], 0); v_cache[li] = v if v_cache[li] is None else np.concatenate([v_cache[li], v], 0); k, v = k_cache[li], v_cache[li]
            kk, vv = np.repeat(k, grp, axis=1), np.repeat(v, grp, axis=1)                  # each key-value head serves `grp` query heads
            sc = np.einsum("qhd,khd->hqk", q, kk) / np.sqrt(hd); T = kk.shape[0]
            sc = sc + np.triu(np.full((L, T), -1e9), k=T - L + 1)                        # causal mask (the new tokens sit at the end of the cache)
            a = np.exp(sc - sc.max(-1, keepdims=True)); a /= a.sum(-1, keepdims=True)
            x = x + self.lin(np.einsum("hqk,khd->qhd", a, vv).reshape(L, h), b["o"], ad, "o")
            y = rms_norm(x, b["n2"]); x = x + self.lin(silu(self.lin(y, b["gate"], ad, "gate")) * self.lin(y, b["up"], ad, "up"), b["down"], ad, "down")
        return rms_norm(x, self.norm) @ self.head

    def merged(self, adapter):                                                            # W' = W + (alpha / r) A B for every adapted matrix
        m = TinyLlama(self.cfg); m.emb, m.head, m.norm = self.emb, self.head, self.norm
        m.blocks = [dict(b) for b in self.blocks]
        for li, lay in enumerate(adapter["layers"]):
            for t, (A, B) in lay.items(): m.blocks[li][t] = self.blocks[li][t] + adapter["alpha"] / adapter["r"] * (A @ B)
        return m

tl = TinyLlama(); ids = np.random.default_rng(5).integers(0, TINY["vocab"], 12)
print(f"tiny model: {tl.n_params():,} parameters; formula {llama_param_count(**TINY):,}"); assert tl.n_params() == llama_param_count(**TINY)
base = tl.forward(ids)
ad0 = tl.new_adapter(zero_B=True);  print("1) a fresh adapter (B = 0) changes nothing: max |difference| =", np.abs(tl.forward(ids, ad0) - base).max()); assert np.abs(tl.forward(ids, ad0) - base).max() == 0
adA = tl.new_adapter(seed=2, zero_B=False); adB = tl.new_adapter(seed=3, zero_B=False)
print("2) a trained-like adapter changes the output: max |difference| =", round(float(np.abs(tl.forward(ids, adA) - base).max()), 3))
merged = tl.merged(adA); print("3) merging the adapter into the weights gives the same logits: max |difference| =", float(np.abs(merged.forward(ids) - tl.forward(ids, adA)).max())); assert np.abs(merged.forward(ids) - tl.forward(ids, adA)).max() < 1e-10
out_A, out_B = tl.forward(ids, adA), tl.forward(ids, adB); print("4) two experts differ:", round(float(np.abs(out_A - out_B).max()), 3), "| the base model is untouched by swapping:", np.array_equal(tl.forward(ids), base)); assert np.abs(out_A - out_B).max() > 1e-3
n_ad = sum(A.size + B.size for lay in adA["layers"] for A, B in lay.values()); print(f"5) adapter parameters: {n_ad:,}; formula {lora_count(TINY, 4):,}"); assert n_ad == lora_count(TINY, 4)
kc, vc = [None] * TINY["layers"], [None] * TINY["layers"]; step = np.concatenate([tl.forward(ids[i:i + 1], adA, kc, vc, start=i) for i in range(len(ids))])
print("6) cached token-by-token decoding equals the full pass: max |difference| =", float(np.abs(step - out_A).max())); assert np.abs(step - out_A).max() < 1e-9

# RoPE: scores depend only on relative positions
qv, kv_ = np.random.default_rng(7).normal(size=(5, 8, 16)), np.random.default_rng(8).normal(size=(5, 8, 16))
s1 = np.einsum("qhd,khd->hqk", rope(qv, np.arange(5.0)), rope(kv_, np.arange(5.0))); s2 = np.einsum("qhd,khd->hqk", rope(qv, np.arange(5.0) + 100), rope(kv_, np.arange(5.0) + 100))
print("7) RoPE: shifting all positions by 100 leaves the attention scores unchanged: max |difference| =", float(np.abs(s1 - s2).max())); assert np.abs(s1 - s2).max() < 1e-9

In [ ]:
# the documented adapter, and what serving experts costs (full Llama-3.1-8B shapes)
cfgL = LLAMA31_8B; base_n = llama_param_count(**cfgL)
rows = []
for label, r, tg in (("r = 16, attention only (q,k,v,o)", 16, ("q", "k", "v", "o")), ("r = 16, all seven linear layers", 16, ("q", "k", "v", "o", "gate", "up", "down")), ("r = 32, all seven linear layers", 32, ("q", "k", "v", "o", "gate", "up", "down"))):
    p = lora_count(cfgL, r, tg); rows.append((label, f"{p / 1e6:.1f} M", f"{100 * p / base_n:.2f} %", f"{2 * p / 1e6:.0f} MB", f"{4 * p / 1e6:.0f} MB", f"{2498 * 2 * p / 1e9:.0f} GB"))
print(pd.DataFrame(rows, columns=["adapter", "parameters", "share of base", "16-bit file", "32-bit file", "2498 experts (16-bit)"]).to_string(index=False))
print("\nthe documentation of the training script says: rank 16, alpha 64 (scale alpha/r = 4), dropout 0, adapters on 'all attention and MLP projection layers', and 'about 80 million trainable parameters (about 1 %)'.")
print("Rank 16 on all seven layers is 41.9 M (0.5 %); 'about 80 M, 1 %' is what RANK 32 would give (83.9 M). The documentation is inconsistent with itself (or the rank is 32); we cannot tell without the code.")
print("The same documentation mentions about 50 GB of storage per expert: 2498 adapters of this size need 0.2-0.4 TB in total, not 125 TB, so that figure cannot refer to adapters.\n")

kv_token = 2 * cfgL["layers"] * cfgL["kv_heads"] * (cfgL["h"] // cfgL["heads"]) * 2                          # keys and values, 16-bit
print(f"Llama-3.1-8B key-value cache: {kv_token:,} bytes per token ({kv_token // 1024} KiB), because only 8 of the 32 attention heads keep keys and values (grouped-query attention)")
for L in (1000, 2048):
    print(f"   a sequence of {L} tokens: {kv_token * L / 1e6:6.0f} MB; a beam of 20 such sequences: {20 * kv_token * L / 1e9:4.1f} GB")
print(f"weights in 16-bit: {2 * base_n / 1e9:.1f} GB -> an A100-40GB (the authors' hardware) holds weights, adapter and a beam of 20; a 16 GB GPU does not even hold the weights in 16-bit")
assert kv_token == 131_072 and abs(lora_count(cfgL, 16) - 41_943_040) == 0 and lora_count(cfgL, 32) == 83_886_080

## 10b. Training arithmetic, from the documented settings
The wiki's summary of the training scripts (second-hand; read with care) gives: **stage 1** (one LoRA adapter on the whole filtered database): 50 epochs, batch 4 per GPU on 4 GPUs (16 per step), learning rate $2\\times10^{-4}$ with 1 % warm-up and cosine decay, 16-bit precision, maximum length 2048 tokens, final checkpoint at step **750,000**. **Stage 2** (each expert continues from the stage-1 adapter, which is kept trainable): 10 epochs, the same learning rate and batch, 2 % warm-up, weight decay 0.01, wall-time allowances that grow with the number of reactions in the cell. We can combine these into estimates (derived; they assume the documented numbers are right).

In [ ]:
steps1, batch, epochs1 = 750_000, 16, 50
passes1 = steps1 * batch                                              # example passes in stage 1
per_epoch = passes1 / epochs1
print(f"stage 1: {steps1:,} steps x {batch} = {passes1 / 1e6:.1f} M example passes; {epochs1} epochs -> {per_epoch:,.0f} examples per epoch, {per_epoch / batch:,.0f} steps per epoch (the wiki mentions 15,000 and, elsewhere, 250,000)")
print(f"the script is said to use len(df)//5 examples per job: that would put the database at about {5 * per_epoch / 1e6:.1f} million reactions (derived; the preprint's remark about a 100x larger database suggested about 1 million)")
N_reactions, epochs2 = 5 * per_epoch, 10
steps2_total = epochs2 * N_reactions / batch
print(f"stage 2: every reaction is used for {epochs2} epochs by exactly one expert -> {epochs2 * N_reactions / 1e6:.1f} M example passes, {steps2_total:,.0f} steps in total: the same as stage 1, so the two stages cost about the same")

def allowance_hours(n):
    return 1 if n < 400 else 2 if n < 800 else 4 if n < 1600 else 8 if n < 3200 else None
rows = []
for n in (200, 400, 836, 1600, 3200):
    steps = int(np.ceil(epochs2 * n / batch)); hrs = allowance_hours(n)
    rows.append((n, f"{steps:,}", f"{hrs} h" if hrs else "manual", f"{3600 * hrs / steps:.0f} s" if hrs else "-"))
print(pd.DataFrame(rows, columns=["reactions in the cell", "steps (10 epochs, batch 16)", "wall-time allowance", "allowance per step"]).to_string(index=False))
print("an expert with 836 reactions (the documentation's example) needs 523 steps; the allowance works out to about half a minute per step of 16 sequences, which looks generous for a LoRA step (we did not measure one), so part of it is presumably loading and queueing time")
assert steps1 * batch == 12_000_000 and int(np.ceil(10 * 836 / 16)) == 523 and abs(steps2_total - steps1) < 1

## 10c. The prompt: what the experts read and write
The training script formats every reaction with one template (as documented): an **Input** section with the product and reactant SMILES, the atom-mapped reaction SMARTS, reagent and solvent SMILES, and the names of product, reactant, reagent and solvent; then a **Response** section with the **reaction procedure**, the **reaction name and classification**, and the **yield and characterization**; the end-of-sequence token closes the text. At inference the model is given the beginning and writes the rest (this is the chain of Figure 2a of the preprint). We implement the template, a builder for the inference prompt (**our reading**: the input section up to the SMARTS line) and a parser for what the model writes, and test the round trip on a record built from the **components of the preprint's showcase reaction** (a Buchwald-Hartwig amination; the text of the procedure below is a placeholder, *not* a model output and not a laboratory procedure).

In [ ]:
FIELDS = [("product", "Product (SMILES):"), ("reactant", "Reactant (SMILES):"), ("smarts", "Reaction SMARTS:"), ("agent", "Reagents (SMILES):"), ("solvent", "Solvent (SMILES):"),
          ("product_name", "Product Name:"), ("reactant_name", "Reactant Name:"), ("agent_name", "Reagent Name:"), ("solvent_name", "Solvent Name:")]
SECTIONS = [("procedure", "Reaction Procedure:"), ("classification", "Reaction Name and Classification:"), ("yields", "Yield and Characterization:")]

def build_training_text(rec):
    lines = ["### Input:"] + [f"{label} {rec[key]}" for key, label in FIELDS] + ["", "### Response:"]
    for key, label in SECTIONS: lines += [label, rec[key], ""]
    return "\n".join(lines).rstrip("\n") + "\n<EOS>"

def build_prompt(product, reactant):                                          # inference: give the first lines, let the model write the rest (our reading)
    return "\n".join(["### Input:", f"Product (SMILES): {product}", f"Reactant (SMILES): {reactant}", "Reaction SMARTS:"])

def parse_completion(text):
    """Split a (full or partial) text in the template's format into its fields."""
    out, key = {}, None; labels = {label: k for k, label in FIELDS + SECTIONS}
    for line in text.replace("<EOS>", "").splitlines():
        s = line.strip()
        if s in ("### Input:", "### Response:"): key = None; continue
        hit = next((lab for lab in labels if s.startswith(lab)), None)
        if hit:
            key = labels[hit]; rest = s[len(hit):].strip(); out[key] = rest
        elif key is not None and s != "":
            out[key] = (out[key] + "\n" + s).strip() if out.get(key) else s
    return out

MOCK = dict(product="CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(cc1)N1CCN(CC1)c1ccccn1",
            reactant="CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(Cl)cc1.C1CN(CCN1)c1ccccn1",
            smarts="CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(Cl)cc1.C1CN(CCN1)c1ccccn1>>CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(cc1)N1CCN(CC1)c1ccccn1",
            agent="O=C([O-])[O-].[Cs+].[Cs+]", solvent="C1COCCO1",
            product_name="isopropyl 2-methyl-2-(4-(4-(4-(pyridin-2-yl)piperazin-1-yl)benzoyl)phenoxy)propanoate",
            reactant_name="isopropyl 2-(4-(4-chlorobenzoyl)phenoxy)-2-methylpropanoate; 1-(pyridin-2-yl)piperazine", agent_name="cesium carbonate", solvent_name="1,4-dioxane",
            procedure="[PLACEHOLDER TEXT, not a model output: a procedure would be written here, with equivalents, order of addition, temperature, time and work-up.]",
            classification="Chloro Buchwald-Hartwig amination", yields="[PLACEHOLDER: yield and characterization]")
text = build_training_text(MOCK); print(text[:700] + "\n...")
back = parse_completion(text)
print("\nround trip (build, then parse) recovers every field:", all(back[k] == MOCK[k] for k in MOCK))
prompt = build_prompt(MOCK["product"], MOCK["reactant"]); print("\ninference prompt (our reading):\n" + prompt)
partial = parse_completion(prompt + " " + MOCK["smarts"] + "\nReagents (SMILES): " + MOCK["agent"]); print("\na partial completion parses too:", sorted(partial))
assert all(back[k] == MOCK[k] for k in MOCK) and set(partial) >= {"product", "reactant", "smarts", "agent"}

## 10d. Diverse beam search: why MOSAIC returns different procedures
The documented generation settings are **20 beams in 2 groups, a diversity penalty of 0.1, and 20 returned sequences** (the standard Hugging Face arguments `num_beams`, `num_beam_groups`, `diversity_penalty`, `num_return_sequences`). Ordinary beam search tends to return near-copies of the single best string. In **diverse beam search** the beams are split into groups that are expanded one after the other; a token that an earlier group has just chosen at this position is penalized for the later groups, so that the groups explore different continuations. We see the effect on a tiny language model with the same two rules (a bigram model over 5 tokens, sequences of length 6).

In [ ]:
rd = np.random.default_rng(4); V5 = 5; Pm = rd.dirichlet(np.ones(V5) * 0.35, size=V5 + 1)        # row V5 = start; very peaked transitions, like a confident language model
def beam_diverse(B, G, lam, T=6):
    per = B // G; groups = [[((), 0.0)] for _ in range(G)]
    for t in range(T):
        chosen = np.zeros(V5)                                                       # how often each token was chosen at this step by earlier groups
        for g in range(G):
            cand = []
            for seq, lp in groups[g]:
                prev = seq[-1] if seq else V5
                for tok in range(V5): cand.append((seq + (tok,), lp + np.log(Pm[prev, tok]), lp + np.log(Pm[prev, tok]) - lam * chosen[tok]))
            cand.sort(key=lambda c: -c[2]); groups[g] = [(s, lp) for s, lp, _ in cand[:per]]
            for s, _ in groups[g]: chosen[s[-1]] += 1
    return sorted([x for gr in groups for x in gr], key=lambda c: -c[1])

def mean_hamming(seqs): return float(np.mean([sum(a != b for a, b in zip(x, y)) for i, x in enumerate(seqs) for y in seqs[i + 1:]]))
rows = []
for label, G, lam in (("ordinary beam search (1 group)", 1, 0.0), ("2 groups, penalty 0.1", 2, 0.1), ("2 groups, penalty 1.0", 2, 1.0), ("2 groups, penalty 3.0", 2, 3.0)):
    res = beam_diverse(10, G, lam); seqs = [s for s, _ in res]
    rows.append((label, len(set(seqs)), len({s[0] for s in seqs}), f"{mean_hamming(seqs):.2f}", f"{res[0][1]:.2f}", f"{np.mean([lp for _, lp in res]):.2f}"))
print(pd.DataFrame(rows, columns=["method (10 beams)", "distinct sequences", "distinct first tokens", "mean pairwise difference", "best log-prob", "mean log-prob"]).to_string(index=False))
d_ord, d_small, d_big = rows[0], rows[1], rows[3]
print(f"\nsplitting the beam into groups does not by itself make the strings different: with a small penalty the two groups can return the same sequences ({d_small[1]} distinct of 10 here, against {d_ord[1]} for ordinary beam search); a large penalty gives {d_big[1]} distinct sequences with more first tokens and a larger mutual difference ({d_big[3]} against {d_ord[3]}), but the mean log-probability falls from {d_ord[5]} to {d_big[5]}.")
print("whether the documented 0.1 is 'large' depends on how peaked the language model is (our toy model is very peaked); for MOSAIC the returned list holds up to 20 sequences from the two groups, and the paper counts the first of them as the expert's one-shot prediction.")

## 11. Your turn
**Exercise 1 (metric).** For a true reagent set $\{A,B\}$ and a predicted set $\{A,C,D\}$, compute $D$. Is it an exact match? What if the prediction is $\{A,B,C,D,E\}$?

**Exercise 2 (cells).** Change `K` in Section 3 from 30 to 10 and to 100. How do the class purity, the number of reactions per cell, and the IVF recall at `nprobe=1` change? Which choice corresponds to the paper's design (about as many cells as reaction classes)?

**Exercise 3 (precision).** Define an F1-style score from the precision and the exact-match rate. At how many pooled experts is it highest in the toy?

**Exercise 4 (confidence).** Give each toy test reaction a "confidence" from its distance (for example, the three terciles). Plot the exact-match rate against it and decide at which distance you would stop trusting the expert.

**Exercise 5 (ranking).** For $N=4$ experts with $M=(3,3,1,2)$ predictions, list the first eight experiments in order. How many experiments are there in total?

**Exercise 6 (storage).** If the real adapters used rank 64 on all linear layers, how large would one adapter and the whole collection be?

In [ ]:
# Starters
# Exercise 1
print("D({A,B} vs {A,C,D}) =", round(D_metric({"A", "C", "D"}, {"A", "B"}), 2), "| D({A,B} vs {A,B,C,D,E}) =", D_metric({"A", "B", "C", "D", "E"}, {"A", "B"}))

# Exercise 5
rk4 = ranking([3, 3, 1, 2]); print("first eight experiments (rank, expert, prediction):", rk4[:8], "| total:", len(rk4))

# Exercise 6
p64 = lora_params(64, ("q", "k", "v", "o", "gate", "up", "down")); print(f"rank 64, all linear layers: {p64 / 1e6:.0f} M parameters = {2 * p64 / 1e6:.0f} MB per adapter, {N_EXPERTS * 2 * p64 / 1e12:.2f} TB for {N_EXPERTS}")

## 12. Self-check

In [ ]:
assert nn_accuracy(ztr, ytr, zte, yte) > nn_accuracy(Xtr, ytr, Xte, yte) + 0.1 and (lgte.argmax(1) == yte).mean() > 0.95
assert sizes.sum() == len(ztr) and fam_pur.mean() > 0.9
assert np.mean(ivf_search(zte, 1)[0] == exact_nn) < np.mean(ivf_search(zte, 5)[0] == exact_nn) and ivf_search(zte, 3)[1].mean() < 0.5 * len(ztr)
r1, r3 = evaluate(1)[0], evaluate(3)[0]
assert r3["reagent"][0] > r1["reagent"][0] and r3["both"][1] > r1["both"][1] and r3["reagent"][1] > r1["reagent"][1]
assert abs(D_metric({"A", "C", "D"}, {"A", "B"}) - 0.5) < 1e-12 and D_metric({"A", "B", "C"}, {"A", "B"}) == 0
assert ex_all == 1.0 and prec_rows[-1][3] < 30 and prec_rows[0][3] > prec_rows[-2][3]
assert table[-1][2] < table[0][2] - 15 and rho < -0.1
assert abs(fail1 - 27.0) < 1e-9 and abs(failn - 5.2) < 1e-9 and abs(g.loc["exact", "reagent"] - 1.92) < 1e-9 and abs(g.loc["exact", "solvent"] - 1.10) < 1e-9
assert abs(eff[("reagent", "exact")][2] - 2.22) < 0.02 and abs(eff[("solvent", "exact")][2] - 1.12) < 0.02
assert ranks == sorted(ranks) and len(set(ranks)) == len(ranks) and 8 not in ranks and 11 not in ranks
assert abs(fixed_point - 64.08) < 0.01 and abs(base_params / 1e9 - 8.03) < 0.01
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Run the authors' code and visualize a prediction

> **Read this first.**
> * **Unlike DirectMultiStep, MOSAIC is not a pip package.** The repository (`batistagroup/MOSAIC`) is a collection of notebooks and scripts. Its README says to *set up the Pistachio database, or use any custom database*, and does not describe a download of weights. A full prediction needs: the **KMN weights**, the **FAISS index** of the Voronoi centres, a **dataframe** derived from the **commercial Pistachio database**, the **Llama-3.1-8B-Instruct weights** (Meta's license), and the **LoRA adapter of at least one expert** (there are 2,498; the Nature paper's code-availability statement mentions GitHub and an archived Zenodo code base). **We could not determine which of these the repository or the archive actually contains**; the cells below check what is present and run what can be run.
> * The authors' repository is under a **CC BY-NC-SA 4.0** license according to a machine-generated summary (read the `LICENSE` file yourself): non-commercial use, share-alike. This notebook contains none of the authors' code; it clones their repository when you run it.
> * **The author of this notebook could not run Part II** (no network, GPU or PyTorch where it was written). The helper functions were tested against stand-in modules only. The code follows the README and a machine-generated wiki of the repository (second-hand information about function names and settings, which may be wrong in details).
> * The authors' prediction script `run_prediction.sh` is written for an **HPC cluster** (it loads a conda module). So Level C below does *not* call it; it reimplements the documented generation step with Hugging Face `transformers` and `peft`. That is **our reimplementation of the documented settings, not the authors' `RunPrediction.py`**.

**Three levels**, from always possible to most demanding:
* **Level A (always runs, CPU):** a *rehearsal of the output format and the visualization* on a **placeholder** record. It is *not* model output: the procedure text is a placeholder and the numbers are illustrative. It checks that the parsing and drawing code works and shows what a real prediction will look like.
* **Level B (needs the authors' routing artifacts):** run the authors' own `predict_rxn(..., run_prediction=False)`, which encodes the query, finds the nearest Voronoi centres and prints the experts, their distances and what reactions each was trained on.
* **Level C (needs a big GPU, Llama weights and an expert adapter):** write procedures with an expert and draw them.

In [ ]:
# the query: the components of the preprint's showcase reaction (a Buchwald-Hartwig amination), written as a reaction SMILES 'reactants>>product'
QUERY = ("CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(Cl)cc1.C1CN(CCN1)c1ccccn1>>"
         "CC(C)OC(=O)C(C)(C)Oc1ccc(cc1)C(=O)c1ccc(cc1)N1CCN(CC1)c1ccccn1")
N_EXPERT = 3
CONFIRM_LLM = False            # set to True to allow Level C to download the 16 GB Llama weights and run the model
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct"       # the identifier used in the documented example configuration (a Hugging Face mirror); the Llama license applies
ADAPTER_ROOT = "Expert_Model_Checkpoints_RSFP"           # folder with Expert_<id>/ subfolders (LoRA adapters)
EXPERT_IDS = []                # fill in after Level B (the experts it selects), e.g. [2266, 2398]
print("query:", QUERY[:90] + "...")

In [ ]:
!pip install -q rdkit faiss-cpu transformers peft accelerate python-Levenshtein

In [ ]:
import importlib, sys, os, json, subprocess, textwrap, time, inspect
from pathlib import Path

print(sys.version.split()[0])
for mod in ("torch", "transformers", "peft", "faiss", "rdkit", "Levenshtein"):
    try:
        m_ = importlib.import_module(mod); print(f"{mod:14s} {getattr(m_, '__version__', 'installed')}")
    except Exception as e:
        print(f"{mod:14s} NOT AVAILABLE ({type(e).__name__})")
try:
    import torch; HAS_GPU = torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if HAS_GPU else "-")
except Exception:
    HAS_GPU = False
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    HAS_RDKIT = True
except Exception:
    HAS_RDKIT = False
print("RDKit (structure drawings):", HAS_RDKIT, "| GPU:", HAS_GPU)
print("note: the README pins old versions (numpy 1.23.5, torch 2.4.1, transformers 4.45.1, peft 0.12.0, trl 0.11.1, faiss 1.7.4, ...). We install unpinned versions above; if the authors' code fails on a version, create an environment with their pins.")

In [ ]:
# get the repository (notebooks, utilities and training scripts)
if not Path("Main_Control.ipynb").exists():
    try:
        if not Path("MOSAIC").exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/batistagroup/MOSAIC.git"], check=True)
        os.chdir("MOSAIC")
    except Exception as e:
        print("could not clone the repository (no network or no git?):", type(e).__name__, "- Level A below still works; Levels B and C need the repository")
print("working directory:", os.getcwd())
for p in sorted(Path(".").iterdir()):
    if not p.name.startswith("."): print(f"   {p.name + ('/' if p.is_dir() else ''):40s} {'' if p.is_dir() else f'{p.stat().st_size / 1e3:9.1f} kB'}")

In [ ]:
# which of the artifacts a prediction needs are actually here?
EXPECTED = {"KMN weights (.pth)": ["**/best_model_50ep*.pth", "Model_Weights/**/*.pth"], "FAISS index of the Voronoi centres": ["**/RSFP_Index.index", "Model_Weights/**/*.index"],
            "expert-assignment dataframe (Pistachio-derived)": ["**/RSFP_Train_*_df.pkl", "Model_Weights/**/*.pkl"], "stage-1 LoRA checkpoint": ["**/checkpoint-750000"],
            "expert LoRA adapters": ["**/Expert_Model_Checkpoints_RSFP/Expert_*", "**/Expert_[0-9]*"], "Pistachio sample (100 reactions)": ["Pistachio_Sample_100.csv"]}
found = {k: sorted({p for pat in pats for p in Path(".").glob(pat)}) for k, pats in EXPECTED.items()}
for k, v in found.items():
    print(f"{'FOUND  ' if v else 'missing'}  {k}" + (f": {len(v)} item(s), e.g. {v[0]} ({v[0].stat().st_size / 1e6:.1f} MB)" if v and v[0].is_file() else (f": {len(v)} item(s), e.g. {v[0]}" if v else "")))
if Path("Model_Weights").exists():
    print("\ncontents of Model_Weights/:")
    for p in sorted(Path("Model_Weights").rglob("*"))[:40]: print(f"   {str(p):60s} {'' if p.is_dir() else f'{p.stat().st_size / 1e6:9.2f} MB'}")
CAN_ROUTE = all(found[k] for k in ("KMN weights (.pth)", "FAISS index of the Voronoi centres", "expert-assignment dataframe (Pistachio-derived)"))
CAN_GENERATE = CAN_ROUTE and bool(found["expert LoRA adapters"])
print(f"\nLevel A (format rehearsal): possible.  Level B (routing with the authors' code): {'possible' if CAN_ROUTE else 'NOT possible with what was found'}.  Level C (generation): {'possible if you have a large GPU and the Llama weights' if CAN_GENERATE else 'NOT possible with what was found (needs expert adapters)'}.")

In [ ]:
# the authors' own API, read from their notebook (so that you can check our description of it)
def show_authors_function(nb_path="Main_Control.ipynb", marker="def predict_rxn", n_lines=40):
    nb = json.load(open(nb_path))
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] == "code" and marker in src:
            lines = src.splitlines(); i = next(k for k, l in enumerate(lines) if marker in l)
            print("\n".join(lines[i:i + n_lines])); return
    print("function not found in", nb_path)
def show_path_lines(nb_path="Main_Control.ipynb"):
    nb = json.load(open(nb_path)); print("lines of the notebook's code that look like file paths (edit them with the 'patch' argument below):")
    for c in nb["cells"]:
        if c["cell_type"] == "code":
            for l in "".join(c["source"]).splitlines():
                if any(w in l for w in ("/global/", ".pth", ".index", ".pkl", "path", "Directory")) and not l.lstrip().startswith("#"): print("   ", l.strip()[:150])
if Path("Main_Control.ipynb").exists():
    show_authors_function(); print(); show_path_lines()

In [ ]:
# ---------- Level A helpers: parse a prediction, draw the reaction scheme, the confidence and the expert's domain ----------
import matplotlib.pyplot as plt, numpy as np
from matplotlib.offsetbox import AnnotationBbox, OffsetImage, TextArea

def draw_molecule(ax, smiles, xy, size=(360, 280), zoom=0.40, edge="0.55"):
    mol = Chem.MolFromSmiles(smiles) if HAS_RDKIT else None
    box = OffsetImage(np.asarray(Draw.MolToImage(mol, size=size)), zoom=zoom) if mol is not None else TextArea(chr(10).join(textwrap.wrap(smiles, 20)), textprops=dict(fontsize=6.5, family="monospace"))
    ax.add_artist(AnnotationBbox(box, xy, frameon=True, bboxprops=dict(edgecolor=edge, lw=1.5, facecolor="white"), pad=0.2))

def draw_reaction_scheme(rec, ax):
    left, _, right = rec.get("smarts", "").partition(">>"); reactants = [s for s in left.split(".") if s]; products = [s for s in right.split(".") if s]
    n = len(reactants) + len(products); xs = np.linspace(0.8, 0.2 + 2.6 * (n - 1) / max(n - 1, 1) + 0.6, n) if n > 1 else [1.0]
    x = 0.0
    for i, s in enumerate(reactants):
        draw_molecule(ax, s, (x, 0)); x += 1.25
        if i < len(reactants) - 1: ax.text(x - 0.62, 0, "+", ha="center", va="center", fontsize=16)
    ax.annotate("", xy=(x + 0.55, 0), xytext=(x - 0.25, 0), arrowprops=dict(arrowstyle="-|>", lw=2)); cond = ", ".join(t for t in (rec.get("agent_name", ""), rec.get("solvent_name", "")) if t)
    ax.text(x + 0.15, 0.20, textwrap.fill(cond or rec.get("agent", ""), 24), ha="center", va="bottom", fontsize=7.5); x += 1.35
    for i, s in enumerate(products): draw_molecule(ax, s, (x, 0), edge="#0072B2"); x += 1.25
    ax.set_xlim(-0.8, x - 0.4); ax.set_ylim(-0.75, 0.75); ax.axis("off")

def draw_confidence(ax, distance):
    bands = [(0, 50, "#009E73", "close (< 50)"), (50, 100, "#8FD1B7", "(< 100)"), (100, 200, "#E69F00", "moderate (100-200)"), (200, 350, "#D55E00", "far (> 200)")]
    for a, b, c, t in bands: ax.barh(0, b - a, left=a, color=c, height=0.5); ax.text((a + b) / 2, 0.42, t, ha="center", fontsize=7)
    if distance is not None: ax.axvline(distance, color="k", lw=2.5); ax.text(distance + 5, -0.55, f"this query: {distance:.1f}", ha="left", fontsize=8, fontweight="bold")
    ax.set(xlim=(0, 350), ylim=(-0.8, 0.9), yticks=[], xlabel="distance to the expert's centre (squared, FAISS units)"); ax.spines["left"].set_visible(False)

def draw_composition(ax, comp):
    names = list(comp)[::-1]; ax.barh(range(len(names)), [comp[n] for n in names], color="#0072B2")
    ax.set_yticks(range(len(names))); ax.set_yticklabels([textwrap.shorten(n, 34) for n in names], fontsize=7); ax.set_xlabel("% of the reactions the expert was trained on")

def show_prediction(rec, distance=None, composition=None, title="", procedure_chars=520):
    fig = plt.figure(figsize=(11, 8.2)); gs = fig.add_gridspec(3, 2, height_ratios=[1.5, 0.9, 1.5])
    ax0 = fig.add_subplot(gs[0, :]); draw_reaction_scheme(rec, ax0); ax0.set_title(title, fontsize=10)
    ax1 = fig.add_subplot(gs[1, 0]); draw_confidence(ax1, distance)
    ax2 = fig.add_subplot(gs[1, 1]); (draw_composition(ax2, composition) if composition else ax2.axis("off"))
    ax3 = fig.add_subplot(gs[2, :]); ax3.axis("off")
    txt = "Procedure: " + textwrap.fill(rec.get("procedure", "-")[:procedure_chars], 120) + chr(10) + chr(10) + "Classification: " + rec.get("classification", "-") + chr(10) + "Yield / characterization: " + textwrap.fill(rec.get("yields", "-")[:200], 120)
    ax3.text(0, 1, txt, va="top", fontsize=8.5, family="monospace"); plt.tight_layout(); return fig
print("helpers defined")

In [ ]:
# ---------- Level A: REHEARSAL on a PLACEHOLDER record (NOT model output) ----------
mock_comp = {"Chloro Buchwald-Hartwig amination": 70.7, "Chloro SNAr reaction": 17.2, "Goldberg-type coupling": 0.3, "all other classes": 11.8}     # illustrative numbers in the style of the preprint's Fig. 1e
fig = show_prediction(MOCK, distance=49.9, composition=mock_comp, title="PLACEHOLDER RECORD for rehearsing the format (not a model prediction; illustrative distance and composition)")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True); fig.savefig(OUT / "mosaic_rehearsal.png", dpi=140); plt.show()
print("saved", OUT / "mosaic_rehearsal.png", "| fields parsed from the placeholder text:", sorted(parse_completion(build_training_text(MOCK))))

In [ ]:
# ---------- Level B: the authors' own routing, through their Main_Control.ipynb ----------
def load_namespace(nb_path="Main_Control.ipynb", patch=None):
    """Execute the code cells of the authors' notebook up to (not including) the first cell that CALLS predict_rxn; returns the namespace (which then holds predict_rxn, the loaded KMN, the index, ...)."""
    sys.path.insert(0, str(Path("PredictionUtils").resolve())); ns = {"__name__": "__main__"}; nb = json.load(open(nb_path))
    for c in nb["cells"]:
        if c["cell_type"] != "code": continue
        src = "".join(c["source"]); lines = [l for l in src.splitlines() if not l.lstrip().startswith(("!", "%"))]
        if any(l.lstrip().startswith("predict_rxn(") or "= predict_rxn(" in l for l in lines): break
        src = chr(10).join(lines)
        for old, new in (patch or {}).items(): src = src.replace(old, new)
        exec(compile(src, "<Main_Control cell>", "exec"), ns)
    return ns

# EDIT: map the paths used in the authors' notebook to the files you have, e.g. {"/global/cfs/.../best_model_50ep_4096batchsize_AdamW.pth": "Model_Weights/best_model_50ep_4096batchsize_AdamW.pth"}
PATCH = {}
if CAN_ROUTE:
    try:
        ns = load_namespace(patch=PATCH); import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            ns["predict_rxn"](QUERY, n_expert=N_EXPERT, verbose=True, run_prediction=False)
        report = buf.getvalue(); print(report[:3000])
    except Exception as e:
        print("running the authors' routing failed:", type(e).__name__, e, "\nlook at the path lines printed above and fill in PATCH; the authors' notebook was written for their cluster")
else:
    print("Level B skipped: the routing artifacts were not found (see the scan above).")

In [ ]:
# read the experts and distances out of the authors' printed report (their format, as documented: 'Expert ID: 2266' and 'Expert Centroid Distances: 49.89'), so that Level C can use them
import re as _re
def parse_expert_report(text):
    ids = [int(x) for x in _re.findall(r"Expert ID:\s*(\d+)", text)]; dist = [float(x) for x in _re.findall(r"Expert Centroid Distances:\s*([0-9.]+)", text)]
    return ids, dist
if "report" in globals():
    ids_found, dist_found = parse_expert_report(report); print("experts selected:", ids_found, "| centroid distances:", dist_found)
    if ids_found and not EXPERT_IDS: EXPERT_IDS = ids_found

In [ ]:
# ---------- Level C: write procedures with an expert (our reimplementation of the documented generation step) ----------
class ExpertRunner:
    """Loads the base model once and applies expert LoRA adapters one at a time (standard transformers + peft; NOT the authors' RunPrediction.py)."""
    def __init__(self, base_model, token=None, load_4bit=False):
        import torch; from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch = torch; self.tok = AutoTokenizer.from_pretrained(base_model, token=token); kw = dict(device_map="auto", token=token)
        if load_4bit: kw["load_in_4bit"] = True
        else: kw["torch_dtype"] = torch.float16
        self.base = AutoModelForCausalLM.from_pretrained(base_model, **kw); self.model = None; self.loaded = set()
    @staticmethod
    def adapter_dir(expert_dir):
        ck = sorted(Path(expert_dir).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])); return ck[-1] if ck else Path(expert_dir)
    def use_expert(self, expert_id, adapter_root):
        from peft import PeftModel
        path = self.adapter_dir(Path(adapter_root) / f"Expert_{expert_id}"); name = f"expert_{expert_id}"
        if self.model is None: self.model = PeftModel.from_pretrained(self.base, str(path), adapter_name=name)
        elif name not in self.loaded: self.model.load_adapter(str(path), adapter_name=name)
        self.loaded.add(name); self.model.set_adapter(name); self.model.eval()
    def generate(self, query, beams=20, groups=2, penalty=0.1, n_return=5, max_new_tokens=900):
        left, _, right = query.partition(">>"); prompt = build_prompt(right, left)
        inputs = self.tok(prompt, return_tensors="pt").to(self.model.device); n_in = inputs["input_ids"].shape[1]
        with self.torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=beams, num_beam_groups=groups, diversity_penalty=penalty, num_return_sequences=n_return, do_sample=False)
        return [prompt + t for t in self.tok.batch_decode(out[:, n_in:], skip_special_tokens=True)]

predictions = {}
if CONFIRM_LLM and EXPERT_IDS:
    runner = ExpertRunner(BASE_MODEL)
    for eid in EXPERT_IDS[:N_EXPERT]:
        t0 = time.time(); runner.use_expert(eid, ADAPTER_ROOT); predictions[eid] = runner.generate(QUERY); print(f"expert {eid}: {len(predictions[eid])} sequences in {time.time() - t0:.0f} s")
elif not CONFIRM_LLM:
    print("Level C skipped: set CONFIRM_LLM = True (in the configuration cell) to download the Llama weights and run the model. It needs about 16 GB of GPU memory for the weights alone plus the beams.")
else:
    print("Level C skipped: EXPERT_IDS is empty. Fill it in (from the routing report of Level B) and make sure", ADAPTER_ROOT, "contains Expert_<id>/ folders.")

In [ ]:
# draw what the experts wrote: the reaction scheme, the confidence and the procedure (REAL model output only if Level C ran)
dist_by_expert = dict(zip(globals().get("ids_found", []), globals().get("dist_found", [])))
for eid, texts in predictions.items():
    for k, text in enumerate(texts[:2], 1):
        rec = parse_completion(text); rec.setdefault("smarts", QUERY)
        fig = show_prediction(rec, distance=dist_by_expert.get(eid), composition=None, title=f"expert {eid}, candidate {k} (written by the model)"); fig.savefig(OUT / f"mosaic_expert{eid}_cand{k}.png", dpi=140); plt.show()
        print(f"expert {eid}, candidate {k}: parsed fields {sorted(rec)}")
if not predictions: print("no model predictions in this session: only the Level A rehearsal above was drawn")

### What to look for in a real prediction
* **Which experts were chosen, and how far are they?** A distance below about 50 means the query lies well inside the expert's domain (the preprint); the Nature summary we read says success rates above 75 % for distances below 100. Look at what the expert was trained on (the composition list): the best expert for a Buchwald-Hartwig query should be dominated by related C-N couplings.
* **Do the candidates agree?** Compare the reagents, solvents, temperatures and times across the beam, and across experts. The preprint's own statistics show that one expert often returns one of several plausible conditions.
* **Is the reaction scheme the query?** The model first writes the atom-mapped reaction; check that it matches your query and that the reagents look sensible.
* **Is the procedure complete?** Equivalents, order of addition, temperature, time, work-up and yield are all part of the text.
* **It is a suggestion, not a laboratory procedure.** The authors state that procedures must only be carried out by trained individuals in properly equipped laboratories; the text may omit hazards.

### If something fails
* **Imports fail or versions clash:** the README pins old versions (numpy 1.23.5, torch 2.4.1, transformers 4.45.1, peft 0.12.0, faiss 1.7.4). Try a fresh environment with those pins.
* **`load_namespace` fails with a file-not-found error:** the authors' notebook uses paths from their cluster; list them (cell above) and fill in `PATCH`.
* **Level B needs the Pistachio-derived dataframe**, which may not be public; without it only Level A can run.
* **Out of memory in Level C:** use a GPU with at least 24 GB, reduce `beams` (it must be divisible by `groups`), or pass `load_4bit=True` (needs `bitsandbytes`).
* **The parsed fields are empty:** print the raw text; the model's output format may differ from the template we assume (our reading of the documented template).